<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# LOGISTIC REGRESSION CLASSIFICATION WITH SCIKIT-LEARN

<br>

**About:** A hands-on walkthrough of multi-class logistic regression using scikit-learn, covering data exploration, model training, performance evaluation, decision boundary visualization, and feature scaling on the Iris dataset.

**Learning Goals:** (1) Understand when logistic regression applies vs. linear regression. (2) Fit a multi-class logistic regression model using sklearn. (3) Compare One-vs-Rest and Softmax (multinomial) strategies. (4) Interpret accuracy, precision, recall, and a confusion matrix. (5) Visualize decision boundaries in 2D feature space. (6) Apply Min-Max and Standard scaling and evaluate the effect on model accuracy.

**Keywords:** logistic regression, classification, scikit-learn, confusion matrix, feature scaling, decision boundaries, multiclass

**Prerequisite Knowledge:** (1) Python and pandas basics, (2) NumPy arrays, (3) Basic probability concepts

**Target User:** Learners who have seen linear regression and want to extend their understanding to classification problems using a real dataset and a production-grade library.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SCIKIT-LEARN OVERVIEW](#Part_0)
> #### [PART 1: UNDERSTANDING THE DATA](#Part_1)
> #### [PART 2: BUILDING THE LOGISTIC REGRESSION MODEL](#Part_2)
> #### [PART 3: MEASURING PERFORMANCE](#Part_3)
> #### [PART 4: VISUALIZING DECISION BOUNDARIES](#Part_4)
> #### [PART 5: FEATURE SCALING](#Part_5)

<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn import linear_model, metrics
from sklearn.metrics import (
    precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SCIKIT-LEARN** OVERVIEW

Scikit-learn provides supervised and unsupervised learning algorithms through a consistent Python interface. The library focuses on modeling - it is not designed for data loading or manipulation (those tasks belong to pandas and NumPy). Every estimator in scikit-learn follows the same three-step pattern: instantiate, fit, predict (or transform). This consistency means the skills you build with logistic regression transfer directly to random forests, SVMs, or any other estimator.

Capability areas include:

- [Regression](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning) - predicting continuous values
- [Classification](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning) - predicting discrete class labels
- [Clustering](https://scikit-learn.org/stable/modules/clustering.html#clustering) - grouping unlabeled data
- [Dimensionality Reduction](https://scikit-learn.org/stable/modules/decomposition.html#decompositions) - compressing feature spaces
- [Model Selection](https://scikit-learn.org/stable/model_selection.html#model-selection) - cross-validation, hyperparameter tuning
- [Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing) - scaling, encoding, imputation

**The general ML workflow** this notebook follows:

1. **Understand the data** - visualize distributions, check for missing values, check class balance.
2. **Preprocess** - encode labels as integers, scale features if needed.
3. **Split** - hold out a test set before touching the model.
4. **Fit** - train the model on training data only.
5. **Evaluate** - measure accuracy, precision, and recall on held-out test data.
6. **Iterate** - try variations (different solvers, scaling) and compare.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **In the three-step sklearn pattern (instantiate, fit, predict), which step must only use training data? Why would using test data in that step give you an overly optimistic accuracy score?**

<br>

```python
# Write your answer as a comment or in a markdown cell below
```

<hr style="border: 2px solid#003262;" />

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **UNDERSTANDING** THE DATA

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/iris.png" align="center" width="40%" padding="10"><br>
    <br>
    The Iris dataset: three species classified by sepal and petal measurements.
</div>

**Problem:** The Iris setosa species has medicinal benefits, and correctly identifying iris species from physical measurements would make that identification scalable. Using sepal length and width measurements, we want to predict which of three species - setosa, versicolor, or virginica - a flower belongs to.

This is a **multi-class classification** problem: the target is a discrete label with more than two possible values. Logistic regression handles this natively through two strategies we will compare in Part 2.

#### CONTENTS:

> [PART 1.1: Load and Inspect](#Part_1_1)<br>
> [PART 1.2: Check Class Balance and Missing Values](#Part_1_2)<br>
> [PART 1.3: Encode Labels and Separate Features](#Part_1_3)<br>
> [PART 1.4: Visualize Feature Distributions](#Part_1_4)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: LOAD AND INSPECT

<br>

We load the dataset with pandas and immediately call `head()` to confirm the column names and data types. This is not optional: it takes 10 seconds and has caught more silent data-loading bugs than any other practice.

In [ ]:
data = pd.read_csv('iris_classification.csv')

# Confirm shape and first rows before doing anything else
print('Shape:', data.shape)
data.head(5)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: CHECK CLASS BALANCE AND MISSING VALUES

<br>

Class imbalance and missing values are two separate problems that both need to be caught before fitting any model.

**Class imbalance:** If one class has 10x more samples than another, a model that always predicts the majority class will achieve high accuracy while learning nothing useful. Balanced classes are the easiest case; if classes are unbalanced, you need to account for that in your evaluation metric choice (accuracy becomes misleading - use F1 or AUC instead).

**Missing values:** Most sklearn estimators cannot handle NaN values and will raise an error. You want to find them before fitting, not during.

In [ ]:
# How many samples per class?
print('Samples per species:')
print(data['species'].value_counts())

# Any missing values?
print('\nNaN count per column:')
print(data.isnull().sum())

___

**Note:** The Iris dataset is perfectly balanced (50 samples per class) with no missing values. This makes it an ideal learning dataset. Real-world datasets rarely have this property.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: ENCODE LABELS AND SEPARATE FEATURES

<br>

Sklearn's `LogisticRegression` expects numeric labels, not strings. We map species names to integers: versicolor=0, virginica=1, setosa=2. The specific numbers do not affect the model - logistic regression treats them as discrete categories, not an ordered scale.

We also shuffle the data before splitting. The original CSV may be sorted by species, which would concentrate one class entirely in the test set if we split without shuffling.

In [ ]:
# Shuffle to prevent any ordering artifacts from the CSV
data = shuffle(data).reset_index(drop=True)

# Separate features (X) and labels (Y)
X = data.iloc[:, :-1]          # all columns except the last
Y = data['species']             # target column

# Encode string labels as integers
Y = Y.map({'versicolor': 0, 'virginica': 1, 'setosa': 2})

print('Feature matrix shape:', X.shape)
print('Label vector shape: ', Y.shape)
print('\nFirst 5 labels after encoding:')
print(Y.head())

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.4: VISUALIZE FEATURE DISTRIBUTIONS

<br>

Looking at feature distributions before modeling tells you two things relevant to logistic regression:

1. **Scale differences** - if sepal_length ranges 4-8 and petal_length ranges 1-7, gradient-based solvers (like lbfgs) converge slower on the unscaled features. You'll address this in Part 5.
2. **Class separability** - plotting per-class histograms reveals which features carry the most discriminative signal. A feature whose distributions overlap heavily across classes is a weak predictor on its own.

In [ ]:
# Summary statistics - confirm scale and range of each feature
data.describe()

In [ ]:
# Overall feature distributions
data.hist(figsize=(15, 3))
plt.suptitle('Feature distributions (all classes combined)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class distributions - which features separate the classes?
print(data.groupby('species').count())

data.groupby('species').hist(figsize=(15, 3))
plt.suptitle('Per-class feature distributions', y=1.02)
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **From the per-class histograms, which single feature appears to separate setosa from the other two species most clearly? Write one or two sentences explaining what you see in the plot that supports your answer.**

<br>

```python
# Your answer (can be a comment or look at the plots above)
```

<hr style="border: 2px solid#003262;" />

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **BUILDING** THE LOGISTIC REGRESSION MODEL

#### CONTENTS:

> [PART 2.1: Train/Test Split](#Part_2_1)<br>
> [PART 2.2: Fit the Model](#Part_2_2)<br>
> [PART 2.3: Training Accuracy](#Part_2_3)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: TRAIN/TEST SPLIT

<br>

Before fitting any model, we hold out a portion of the data as a **test set**. The model never sees this data during training. Evaluating on held-out data gives an honest estimate of how the model will perform on new observations - the thing you actually care about.

We use an 80/20 train/test split and fix `random_state=100` so results are reproducible across runs.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/splitting-data.PNG" align="center" width="50%" padding="10"><br>
    <br>
    Training data fits the model. Test data evaluates it on unseen examples.
</div>

___

**Note:** There is no separate validation set here because we are not tuning hyperparameters. When tuning (e.g., choosing the regularization strength C), you need a third split - or use cross-validation - to avoid using the test set for both selection and final evaluation.

___

In [ ]:
# 80% training, 20% test - fixed seed for reproducibility
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=100
)

print('Samples in training set:', len(x_train))
print('Samples in test set:    ', len(x_test))

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: FIT THE MODEL

<br>

When there are more than two classes, logistic regression needs a strategy for extending the binary sigmoid to multi-class probabilities. Sklearn's `LogisticRegression` offers two:

<br>

**One-vs-Rest (OvR)**

Trains one binary classifier per class - "Is this setosa vs. not setosa?", "Is this versicolor vs. not versicolor?", etc. Each classifier outputs a probability, and the class with the highest probability wins. Probabilities from separate classifiers are not constrained to sum to 1.

<br>

**Softmax Regression (multinomial)**

Trains a single model that outputs a probability distribution over all classes simultaneously via the softmax function. Probabilities always sum to 1 by construction. This is the generalization of the binary sigmoid to K classes:

$$P(y = k | x) = \frac{e^{\theta_k^T x}}{\sum_{j=1}^{K} e^{\theta_j^T x}}$$

where $\theta_k$ is the weight vector for class $k$. Each term in the denominator is the unnormalized score for one class; dividing by the sum normalizes so all K probabilities sum to 1. The model is trained end-to-end by minimizing cross-entropy loss across all K classes simultaneously.

___

**Note:** The `solver` parameter controls the optimization algorithm. `newton-cg` uses second-order curvature information and generally converges faster than gradient descent on small datasets. Other solvers (lbfgs, saga) are available - choice depends on dataset size and regularization type. The `multi_class` parameter is deprecated in newer sklearn versions (>=1.5) in favor of always using multinomial when the solver supports it. Code below uses it explicitly for clarity.
# TODO: verify multi_class parameter behavior against current sklearn docs at https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

___

In [ ]:
# Fit a multinomial (Softmax) logistic regression
# newton-cg solver supports multinomial natively
LogisticRegressionModel = linear_model.LogisticRegression(
    solver='newton-cg',
    multi_class='multinomial',
    max_iter=1000
)

LogisticRegressionModel.fit(x_train, y_train)
print('Model fitted successfully')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.3: TRAINING ACCURACY

<br>

Training accuracy tells you how well the model fits the data it was shown. A very high training accuracy with low test accuracy signals overfitting - the model memorized the training set instead of learning generalizable patterns. On the Iris dataset, both training and test accuracy should be close because the dataset is small and well-structured.

In [ ]:
# Training accuracy - how well does the model fit its training data?
training_accuracy = LogisticRegressionModel.score(x_train, y_train)
print('Training Accuracy:', training_accuracy)

# Show manually how accuracy is computed from predictions
predicted_label = LogisticRegressionModel.predict(x_train)
correct = (predicted_label == y_train).sum()
print(f'Correctly classified: {correct} / {len(y_train)} = {correct / len(y_train):.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **If training accuracy is 0.99 and test accuracy is 0.65, what does this gap tell you about the model? What is the name for this failure mode, and name one thing you could do to reduce it.**

<br>

```python
# Write your answer as a comment
```

<hr style="border: 2px solid#003262;" />

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **MEASURING** PERFORMANCE

#### CONTENTS:

> [PART 3.1: Test Accuracy and Why It Is Not Enough](#Part_3_1)<br>
> [PART 3.2: Precision and Recall](#Part_3_2)<br>
> [PART 3.3: Confusion Matrix](#Part_3_3)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: TEST ACCURACY AND WHY IT IS NOT ENOUGH

<br>

Test accuracy answers: "out of all test samples, what fraction did the model predict correctly?" That is useful, but it hides important failure patterns.

Consider a binary classifier on a dataset where 95% of samples are class A and 5% are class B. A model that always predicts class A achieves 95% accuracy without learning anything. Accuracy is a reasonable summary metric when classes are balanced and all errors are equally costly - neither of which holds in most real problems.

In [ ]:
# Test accuracy on held-out data
test_accuracy = LogisticRegressionModel.score(x_test, y_test)
print('Test Accuracy:', test_accuracy)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: PRECISION AND RECALL

<br>

The four building blocks of classification error analysis:

- **True Positive (TP)** - model predicted class X, actual label is class X (correct)
- **True Negative (TN)** - model predicted not-X, actual label is not-X (correct)
- **False Positive (FP)** - model predicted class X, actual label is not-X (wrong)
- **False Negative (FN)** - model predicted not-X, actual label is class X (wrong)

<br>

**Accuracy** - fraction of all predictions that were correct:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

<br>

**Precision** - of all predicted positives, how many are actually positive? Answers: "when the model says X, how much should I trust it?"

$$\text{Precision} = \frac{TP}{TP + FP}$$

<br>

**Recall** - of all actual positives, how many did the model catch? Answers: "how many real X instances did I miss?"

$$\text{Recall} = \frac{TP}{TP + FN}$$

Precision and recall trade off against each other. A model that never predicts X has perfect precision (no false positives) but zero recall (misses every actual positive). The right trade-off depends on the cost of each error type in your application.

In [ ]:
y_pred = LogisticRegressionModel.predict(x_test)

# Full classification report: precision, recall, F1 per class
print(classification_report(y_test, y_pred))

In [ ]:
# Precision and recall per class as arrays
print("Precision per class:", precision_score(y_test, y_pred, average=None))
print("Recall per class:   ", recall_score(y_test, y_pred, average=None))

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: CONFUSION MATRIX

<br>

A confusion matrix puts every test prediction into a grid where rows are actual classes and columns are predicted classes. Correct predictions appear on the diagonal; off-diagonal cells show specific misclassifications.

This matters because "which classes is the model confusing?" is a different question from "how accurate is the model?" Two models with identical accuracy can have completely different confusion patterns - one might perfectly classify two classes and fail entirely on a third, while the other makes evenly distributed errors.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/confusion-matrix.PNG" align="center" width="40%" padding="10"><br>
    <br>
    Confusion matrix layout: rows = actual, columns = predicted.
</div>

In [ ]:
# Confusion matrix as a labeled DataFrame
y_true = y_test
y_pred = LogisticRegressionModel.predict(x_test)

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    columns=['Predicted 0', 'Predicted 1', 'Predicted 2'],
    index=['Actual 0', 'Actual 1', 'Actual 2']
)
print('Confusion matrix:\n', cm_df)

In [ ]:
# Visual confusion matrix using sklearn's built-in display
disp = ConfusionMatrixDisplay.from_estimator(LogisticRegressionModel, x_test, y_test)
disp.figure_.suptitle('Confusion Matrix')
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Using the confusion matrix output above: identify a pair of classes (if any) that the model confuses most often. Given what you know about the Iris dataset features, propose a hypothesis for why those two classes are harder to separate than the others.**

<br>

```python
# Write your answer as a comment
```

<hr style="border: 2px solid#003262;" />

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **VISUALIZING** DECISION BOUNDARIES

#### CONTENTS:

> [PART 4.1: Build the Mesh](#Part_4_1)<br>
> [PART 4.2: Plot Decision Regions with Training Points](#Part_4_2)<br>
> [PART 4.3: Plot Decision Regions with Test Points](#Part_4_3)<br>

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: BUILD THE MESH

<br>

We can visualize the decision boundaries in 2D because the Iris dataset uses only two features (sepal_length, sepal_width). The approach: create a fine grid of fictitious data points that covers the entire feature space, then predict a class for each point. Coloring the grid by predicted class reveals the decision regions - the zones where the model assigns each label.

This is a visualization technique, not part of the model fitting. The mesh step can be computationally expensive on high-resolution grids; `h=0.02` gives a reasonable balance between resolution and speed.

In [ ]:
# Create a dense grid over the feature space
h = 0.02  # grid step size - smaller = finer resolution, slower computation

x_min, x_max = X['sepal_length'].min() - 0.5, X['sepal_length'].max() + 0.5
y_min, y_max = X['sepal_width'].min() - 0.5, X['sepal_width'].max() + 0.5

sepal_length_range = np.arange(x_min, x_max, h)
sepal_width_range = np.arange(y_min, y_max, h)

# meshgrid creates all combinations of (sepal_length, sepal_width) in the grid
sepal_length_values, sepal_width_values = np.meshgrid(sepal_length_range, sepal_width_range)

# Predict class for every grid point
# np.c_ stacks two 1D arrays as columns into a 2D array
predicted_species = LogisticRegressionModel.predict(
    np.c_[sepal_length_values.ravel(), sepal_width_values.ravel()]
)
print('Grid predictions complete. Points predicted:', len(predicted_species))

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: PLOT DECISION REGIONS WITH TRAINING POINTS

<br>

In [ ]:
# Helper function: draw colored decision regions on an axis
def plot_decision_boundaries(ax):
    predicted_reshaped = predicted_species.reshape(sepal_length_values.shape)
    ax.pcolormesh(sepal_length_values, sepal_width_values, predicted_reshaped, cmap=plt.cm.Paired)
    ax.set_xlabel('Sepal length')
    ax.set_ylabel('Sepal width')

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(15, 7))

# Left: actual training labels overlaid on decision regions
plot_decision_boundaries(ax[0])
ax[0].scatter(x_train['sepal_length'], x_train['sepal_width'],
              c=y_train, edgecolors='k', cmap=plt.cm.Paired)
ax[0].set_title('Training data - actual labels on decision regions')

# Right: predicted training labels overlaid on decision regions
plot_decision_boundaries(ax[1])
ax[1].scatter(x_train['sepal_length'], x_train['sepal_width'],
              c=predicted_label, edgecolors='k', cmap=plt.cm.Paired)
ax[1].set_title('Training data - predicted labels on decision regions')

plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.3: PLOT DECISION REGIONS WITH TEST POINTS

<br>

The test plot is the honest diagnostic: points that fall in the wrong color region are misclassifications on unseen data. If you see a cluster of misclassified points, that tells you where in feature space the model's decision boundary is poorly placed.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(15, 7))

# Left: actual test labels
plot_decision_boundaries(ax[0])
ax[0].scatter(x_test['sepal_length'], x_test['sepal_width'],
              c=y_test, edgecolors='k', cmap=plt.cm.Paired)
ax[0].set_title('Test data - actual labels')

# Right: model predictions on test data
plot_decision_boundaries(ax[1])
ax[1].scatter(x_test['sepal_length'], x_test['sepal_width'],
              c=LogisticRegressionModel.predict(x_test), edgecolors='k', cmap=plt.cm.Paired)
ax[1].set_title('Test data - predicted labels')

plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **In the test plot, do any misclassified points appear in recognizable locations (e.g., near a boundary between two regions)? What does that tell you about the model's decision boundary in that area of feature space?**

<br>

```python
# Write your answer as a comment after examining the plots
```

<hr style="border: 2px solid#003262;" />

<a id='Part_5'></a>

<hr style="border: 2px solid#003262;" />

#### PART 5

## **FEATURE SCALING** AND ITS EFFECT ON ACCURACY

#### CONTENTS:

> [PART 5.1: Why Scale?](#Part_5_1)<br>
> [PART 5.2: Apply Standard Scaling and Compare](#Part_5_2)<br>

<a id='Part_5_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 5.1: WHY SCALE?

<br>

Gradient-based and distance-based algorithms are sensitive to the numerical scale of features. If sepal_length varies from 4 to 8 and petal_length varies from 1 to 7, gradient updates are dominated by the feature with larger magnitude - even though both may be equally predictive.

Scaling removes this bias:

<br>

**Min-Max Scaler** - compresses each feature to the range [0, 1]:

$$X_{\text{scaled}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

Preserves the shape of the distribution. Sensitive to outliers (one extreme value compresses all other values toward a narrow range).

<br>

**Standard (Z-score) Scaler** - transforms to mean 0 and standard deviation 1:

$$z = \frac{x - \mu}{\sigma}$$

where $\mu$ is the column mean and $\sigma$ is the standard deviation. Less sensitive to outliers than Min-Max. The transformed features can extend beyond any fixed range.

___

<strong style="color:red">KEY CONSIDERATION:</strong> Always fit the scaler on training data only, then apply the same transform to test data. Fitting on the full dataset leaks test statistics (min, max, mean, std) into the scaler, which are then baked into the training features - a subtle form of data leakage that artificially inflates test accuracy.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_5_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 5.2: APPLY STANDARD SCALING AND COMPARE

<br>

In [ ]:
# Fit scaler on training data ONLY - do not use test data here
scaler = StandardScaler()
scaler.fit(x_train)               # learns mean and std from training set only
X_train_scaled = scaler.transform(x_train)
X_test_scaled = scaler.transform(x_test)  # applies same transform to test

# Verify: scaled training data should have mean ~0 and std ~1
scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print('After scaling (training set):')
print(scaled_df.describe().loc[['mean', 'std'], :])

In [ ]:
# Fit a new model on scaled features
LogisticRegressionModel_scaled = linear_model.LogisticRegression(
    solver='newton-cg',
    multi_class='multinomial',
    max_iter=1000
)
LogisticRegressionModel_scaled.fit(X_train_scaled, y_train)

# Compare accuracy before and after scaling
train_acc_orig = LogisticRegressionModel.score(x_train, y_train)
test_acc_orig = LogisticRegressionModel.score(x_test, y_test)

train_acc_scaled = LogisticRegressionModel_scaled.score(X_train_scaled, y_train)
test_acc_scaled = LogisticRegressionModel_scaled.score(X_test_scaled, y_test)

print(f'Unscaled  - Train: {train_acc_orig:.4f}, Test: {test_acc_orig:.4f}')
print(f'Scaled    - Train: {train_acc_scaled:.4f}, Test: {test_acc_scaled:.4f}')

___

**Note:** Scaling does not significantly change accuracy here because the Iris dataset features are already on similar scales and the logistic regression loss function is convex regardless. Scaling matters more for: (1) gradient descent convergence speed, (2) regularization fairness (the L2 penalty penalizes large weights, which are inflated by large-scale features), and (3) distance-based models (k-NN, SVM). For logistic regression on well-conditioned data, the difference is often small.

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You have a dataset where one feature ranges from 0 to 1,000,000 and another ranges from 0 to 1. You fit a StandardScaler on the training set and plan to apply it to a new production batch. A colleague suggests fitting the scaler on the combined training + production batch "for better estimates of mean and std." Explain why this is wrong, and what specific problem it introduces.**

<br>

```python
# Write your answer as a comment
```

<hr style="border: 2px solid#003262;" />

<hr style="border: 6px solid#003262;" />